# End to end 3 — Surface: CCD → fit → ascent → constrained allocation → next probe

**The question.** Two treatments, `a` and `b`, with saturating responses. Where should the
next dose-finding study put its arms, what does the fitted surface say the best allocation is
under a budget, and which probe would most sharpen the answer?

Classical response-surface methodology, composed from `axiom.surface`, `axiom.sim`,
`axiom.design`, and `axiom.diagnose`:

1. **Design** a rotatable central composite (`central_composite`) on the two treatments.
2. **Run** it — here, simulate the arms with `arms_world`, whose truth we then pretend not to know.
3. **Fit** with the Laplace backend and compare the posterior to the truth.
4. **Move**: `canonical_analysis` at the design centre, then `steepest_ascent` along the gradient.
5. **Allocate** under a total-dose budget with `allocate`; trace the effort `frontier`.
6. **Probe**: `design_to_identify` for the parameter the fit learned least, and
   `optimal_exchange` for a D-optimal follow-up averaged over posterior draws.
7. **Check**: `posterior_predictive` and `weak_identification` on the fit.

Everything on the surface — the likelihood, the simulator, the design criteria, and the
optimizer — evaluates the same `forward()`.

In [ ]:
import numpy as np

from axiom.core import is_failure
from axiom.design import IdentifyingDesign, design_to_identify
from axiom.diagnose import PPCResult, WeakIdReport, posterior_predictive, weak_identification
from axiom.sim import arms_world
from axiom.surface import (
    Allocation, AscentPath, Bounds, Design, Frontier, HillKernel, Surface, allocate, bayesian_criterion,
    canonical_analysis, central_composite, d_criterion, fit, frontier, full_factorial, optimal_exchange,
    steepest_ascent,
)

from IPython.display import display
from axiom.display import enable, table
from axiom.viz import response_curve

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, ORANGE, annotate, caption, curve_band, heat, mark_x, points

enable();  # every axiom result renders itself from here on

SEED = 0

## 1. The design

A rotatable CCD on the box `a ∈ [0, 100]`, `b ∈ [0, 40]`: the `2²` cube, four axial points at
`α = 2^{1/4}`, and centre replicates. A dose cannot be negative, so the design is *inscribed*:
the axial points land on the box and the cube is scaled inward by `1/α`. Each row of
`Design.as_frame()` is one arm.

In [ ]:
bounds = Bounds(treatments=("a", "b"), low=(0.0, 0.0), high=(100.0, 40.0))
ccd: Design = central_composite(bounds, alpha="rotatable", center_points=4, inscribed=True)
print(ccd.kind, "| arms:", ccd.n, "|", ccd.detail)
display(ccd.as_frame().round(1))

In [ ]:
doses = ccd.doses()
fig = points(
    {"central composite arms": (doses["a"], doses["b"])},
    title="Twelve arms, chosen before anything is run",
    subtitle="the inscribed rotatable CCD on the two-treatment dose box",
    x_title="dose of a", y_title="dose of b",
    height=400,
)
caption(fig, "Corners for the interaction, axial points for the curvature, replicated centre "
             "points for the pure-error estimate. Every arm is a unit-period somebody has to "
             "pay for, which is why the arrangement is a decision rather than a grid.")

## 2. Run it (in a world with known truth)

`arms_world` takes the design's columns as explicit dose vectors — one period, one row per
arm, a shared intercept. Both treatments have Hill responses; `a` saturates at about the
centre of its range, `b` well below its. The truth is recorded so the fit can be judged.

In [ ]:
truth = {"alpha": 2.0, "beta_a": 10.0, "k_a": 50.0, "s_a": 2.0, "beta_b": 6.0, "k_b": 12.0, "s_b": 1.5}
arms = arms_world(
    n_units=ccd.n, treatments=("a", "b"),
    kernels={"a": HillKernel(reference_dose=50.0, amplitude_scale=10.0), "b": HillKernel(reference_dose=15.0, amplitude_scale=10.0)},
    doses=ccd.doses(), truth=truth, noise_sd=0.5, seed=SEED,
)
print(arms.panel)
print("realized noise sd:", round(float(arms.noise.std()), 3))

## 3. Fit

Laplace: mode search plus the Hessian at the mode. If either fails the result is a typed
`Unverified`, not a posterior. Posterior means are close to the truth for the amplitudes and
the half-saturation of `b`; `k_a` and `s_a` are less certain — the CCD has only five distinct
levels of `a`, and the `k`–`s`–`beta` ridge of a Hill kernel is the classic weak direction.

In [ ]:
res = fit(arms.spec, arms.panel, backend="laplace", draws=2000, seed=SEED)
print("converged:", res.converged)
post = res.posterior
assert not is_failure(post)
rows = []
for name in ("alpha", "beta_a", "k_a", "s_a", "beta_b", "k_b", "s_b", "sigma"):
    s = post.summary(name, definition="hdi", mass=0.9)
    t = truth.get(name, float(arms.noise.std()))
    rows.append([name, f"{t:.2f}", f"{s.mean:.2f}", str(s.interval)])
table(rows, headers=("parameter", "truth", "posterior mean", "90% HDI"))
theta_hat = {name: float(post.summary(name).mean) for name in post.names()}
surface: Surface = res.surface

In [ ]:
response_curve(res, "a", n_grid=25, mass=0.9)

## 4. Where does the surface go up?

`canonical_analysis` evaluates the gradient and Hessian of `forward()` at a point, finds the
stationary point of the local quadratic, and classifies it by eigenvalue signs. A saturating
surface has no interior optimum: the local quadratic's stationary point is either absent
(a typed `Unsupported`) or sits out on the plateau, beyond the design box with curvature near
zero. Either way it is not a target — the honest reading is "keep going up", and the cue is
to follow the gradient instead. `steepest_ascent` walks uphill within the bounds and records
the path; on a monotone surface it ends at the corner, which is why the next step needs a
budget.

In [ ]:
centre = {"a": 50.0, "b": 20.0}
sp = canonical_analysis(surface, theta_hat, centre)
print("canonical analysis at the centre ->", sp if is_failure(sp) else (sp.kind, sp.point, np.round(sp.eigenvalues, 4)))
path = steepest_ascent(surface, theta_hat, {"a": 10.0, "b": 2.0}, step=5.0, n_steps=30, bounds=bounds)
assert isinstance(path, AscentPath)
print(f"ascent: {path.n} steps, stopped because {path.stop!r}; best point {path.best()} value {path.values[-1]:.3f}")

In [ ]:
a_grid = np.linspace(0.0, 100.0, 21)
b_grid = np.linspace(0.0, 40.0, 17)
mesh = [[float(np.ravel(surface.forward({"a": np.array([av]), "b": np.array([bv])}, theta_hat))[0])
         for av in a_grid] for bv in b_grid]
fig = heat(
    mesh, [f"{v:.0f}" for v in a_grid], [f"{v:.0f}" for v in b_grid],
    text_fmt="", colorbar_title="expected outcome",
    title="The fitted surface, and the walk up it",
    subtitle="expected outcome over the dose box at the posterior mean, with the ascent path",
    x_title="dose of a", y_title="dose of b",
    height=460,
)
fig.add_scatter(x=[p[0] for p in path.points], y=[p[1] for p in path.points], mode="lines+markers",
                line={"color": "#ffffff", "width": 2}, marker={"size": 7, "color": "#ffffff"},
                name="ascent path", showlegend=True)
caption(fig, "No interior peak — the surface rises to a corner and flattens, which is what "
             "'saturating in both treatments' looks like. The ascent walks there and stops at "
             "the box, and that is exactly why the next section imposes a budget instead.")

## 5. Allocation under a budget

Unconstrained, the ascent goes to the corner — more dose is always better on a monotone
surface. The real question has a budget. `allocate` maximizes the expected outcome subject to
`a + b ≤ budget` and the box (SLSQP); a failed solve is `Unsupported`, never a number. Pass the
*posterior* rather than point estimates and the objective is the posterior-mean outcome;
`frontier` traces the allocation over budgets and its shadow prices say what one more unit of
dose is worth at each.

Because the world is known, the fitted allocation can be scored on the true surface.

In [ ]:
BUDGET = 60.0
alloc = allocate(surface, post, budget=BUDGET, bounds=bounds, objective="mean", seed=SEED)
assert isinstance(alloc, Allocation)
print("fitted allocation:", {k: round(v, 2) for k, v in alloc.doses.items()}, f"expected outcome {alloc.expected_outcome:.3f} ({alloc.status}, total dose {alloc.total_dose:.1f})")

oracle = allocate(arms.surface, arms.theta, budget=BUDGET, bounds=bounds, objective="mean", seed=SEED)
assert isinstance(oracle, Allocation)
true_at_fitted = float(arms.forward({k: float(v) for k, v in alloc.doses.items()}).mean())
print("oracle allocation:", {k: round(v, 2) for k, v in oracle.doses.items()}, f"true outcome {oracle.expected_outcome:.3f}")
print(f"true outcome at the fitted allocation: {true_at_fitted:.3f}  (regret {oracle.expected_outcome - true_at_fitted:.3f})")

In [ ]:
fr = frontier(surface, theta_hat, budgets=[20.0, 40.0, 60.0, 90.0, 140.0], bounds=bounds, seed=SEED)
assert isinstance(fr, Frontier)
display(fr.as_frame().round(3))
print("shadow prices:", np.round(fr.shadow_prices(), 4))

In [ ]:
frame = fr.as_frame()
fig = curve_band(
    frame["budget"], frame["expected_outcome"],
    label="best achievable",
    title="What the next unit of budget is worth",
    subtitle="the effort frontier: expected outcome under an optimal split at each budget",
    x_title="total dose budget", y_title="expected outcome",
)
mark_x(fig, BUDGET, text=f"the budget on the table ({BUDGET:.0f})")
annotate(fig, float(frame["budget"].iloc[-1]), float(frame["expected_outcome"].iloc[-1]),
         f"shadow price {fr.shadow_prices()[-1]:.3f}")
caption(fig, "The curve flattens because both treatments saturate. The shadow price at the "
             "right-hand end is what a unit of dose buys there, and it is the number that "
             "says when to stop asking for more budget.")

## 6. The next probe

Two complementary questions. **Which single parameter is least learned, and which arms would
fix that?** `design_to_identify` picks `n` rows from a candidate grid (replicates allowed) to
minimize the expected posterior sd of one target under Gaussian priors set from the current
posterior. **Which follow-up design is best for the whole surface?** `optimal_exchange` runs
Fedorov point exchange for the D-criterion on the linearized surface, averaged over posterior
draws so the optimum is not local to one parameter value (`bayesian_criterion`). Expect it to
push arms to the corners of the box and replicate them — that is what D-optimality on a
linearization rewards — which is a different answer from the CCD's, and from the single-target
probe's; the analyst chooses which question the next study is for.

In [ ]:
sds = {name: float(post.summary(name).sd) for name in post.names()}
target = max((n for n in sds if n != "sigma"), key=lambda n: sds[n] / abs(theta_hat[n]))
print(f"least-learned parameter (relative sd): {target} (sd {sds[target]:.3f}, mean {theta_hat[target]:.3f})")
grid = full_factorial(bounds, 5)
prior_sds = {n: v for n, v in sds.items() if n != "sigma"}
theta_structural = {n: v for n, v in theta_hat.items() if n != "sigma"}
probe = design_to_identify(surface, grid, theta_structural, theta_hat["sigma"], target=target, n=6,
                           prior_sds=prior_sds, seed=SEED, method="finite")
assert isinstance(probe, IdentifyingDesign)
print("chosen arms (a, b):", [tuple(round(x, 1) for x in p) for p in probe.design.points])
print(f"expected sd of {target} after those arms: {probe.expected_sd:.3f} (now {sds[target]:.3f})")

In [ ]:
rng = np.random.default_rng(SEED)
idx = rng.choice(post.n_draws(), size=8, replace=False)
draws = [{name: float(post.flat(name)[i]) for name in post.names()} for i in idx]
follow_up = optimal_exchange(surface, grid, n=ccd.n, theta_draws=draws, criterion="d", seed=SEED)
assert isinstance(follow_up, Design)
print("D-optimal follow-up:", [tuple(round(x, 1) for x in p) for p in follow_up.points])
print(f"Bayesian D: CCD {bayesian_criterion(surface, ccd, draws):.3f} -> follow-up {bayesian_criterion(surface, follow_up, draws):.3f}")
print("local D at the posterior mean: CCD", round(d_criterion(surface.linearize(ccd.doses(), theta_hat)), 3),
      "| follow-up", round(d_criterion(surface.linearize(follow_up.doses(), theta_hat)), 3))

## 7. Trust checks

Before the allocation is acted on: does the fitted model reproduce the observed outcome's
summary statistics (`posterior_predictive`, each statistic against its predictive interval
and tail probability), and is the posterior geometry healthy (`weak_identification`: the
correlation ridge, condition number, and parameters the data did not move)? Flagged
`beta`–`k`–`s` pairs are expected from a twelve-arm CCD on two Hill kernels; they are the
directions the probes in step 6 add arms to, and the reason `passed` is `False` is the reason
a next study exists.

In [ ]:
ppc = posterior_predictive(res, n_draws=100, seed=SEED)
assert isinstance(ppc, PPCResult)
table(
    [
        [s.name, f"{s.observed:.3f}", f"[{s.interval.lower:.3f}, {s.interval.upper:.3f}]",
         f"{s.p_two_sided:.3f}", str(s.extreme)]
        for s in ppc.statistics
    ],
    headers=("statistic", "observed", "predictive interval", "two-sided p", "extreme"),
)
print("extreme statistics:", ppc.extreme_statistics or "none")

rep = weak_identification(res, rho_threshold=0.9)
assert isinstance(rep, WeakIdReport)
print("ridge pairs:", [(p, q, round(rho, 3)) for p, q, rho in rep.high_pairs])
print("condition number:", round(rep.condition_number, 1), "| unlearned:", rep.unlearned, "| saturated:", rep.saturated, "| passed:", rep.passed)

## What this notebook decided

- A 12-arm inscribed CCD was enough to fit both Hill responses with the truth inside every
  90 % interval; the shape parameters of both kernels sit on the usual `beta`–`k`–`s` ridges.
- The surface has no interior optimum, so canonical analysis correctly declines and the
  ascent path is the guide; under a budget, `allocate` gives the split and the frontier's
  shadow prices say what the next unit of dose is worth.
- The fitted allocation's regret on the true surface is small — and known, because the world
  carries its truth.
- The next study should put its arms where `design_to_identify` and `optimal_exchange` say,
  which is where the weak-identification report says the information is missing.